# LeetCode 634 — Find the Derangement of An Array bằng Python

Link đề: https://leetcode.com/problems/find-the-derangement-of-an-array/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def findDerangement(self, n: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium**. Đề trong notebook được viết theo hiểu biết về bài (đối chiếu với bản mô tả công khai), nên nếu bạn có tài khoản Premium hãy đối chiếu lại đề gốc, đặc biệt phần ví dụ và ràng buộc. Mọi đáp án trong notebook đều được kiểm chứng bằng code (vét cạn + so sánh nhiều cách + dãy số đã biết).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **hoán vị "lệch hết"** (derangement) của dãy `1, 2, ..., n` — hoán vị mà **không phần tử nào đứng đúng vị trí ban đầu** — rồi lấy kết quả theo modulo `10^9 + 7`.
2. Nắm các công cụ Python: `itertools.permutations`, đệ quy quay lui, `math.factorial`, `%` (modulo) và tính chất `(a + b) % M`, `(a * b) % M`, gán nhiều biến đồng thời `a, b = b, a + b`.
3. Đi từ **liệt kê mọi hoán vị** → **quay lui có cắt tỉa** → **công thức bao hàm – loại trừ** → **quy hoạch động `f[i] = (i - 1) * (f[i - 1] + f[i - 2])`** → **công thức một số hạng `f[i] = i * f[i - 1] + (-1)^i`**.
4. Hiểu mấu chốt: đặt **số 1 vào vị trí `j` bất kỳ** (có `i - 1` cách), rồi xét số `j` có quay về vị trí 1 hay không — mỗi nhánh quy về một bài **nhỏ hơn cùng dạng** (đếm theo công thức truy hồi).
5. Tránh các lỗi kinh điển: quên lấy modulo, sai trường hợp cơ sở (`f[0] = 1`, `f[1] = 0`), nhân với `i` thay vì `i - 1`, đệ quy sâu `10^6` tầng (RecursionError), cập nhật hai biến sai thứ tự, và dùng số thực (`n!/e`) cho `n` lớn.

## 2. Đề bài & ví dụ

Trong toán tổ hợp, **derangement** (hoán vị lệch hết) là một hoán vị các phần tử của một tập sao cho **không phần tử nào xuất hiện ở vị trí ban đầu của nó**.

Cho số nguyên `n`. Ban đầu có mảng gồm `n` số nguyên từ `1` đến `n` theo thứ tự tăng dần. Hãy trả về **số derangement** mà mảng này có thể tạo ra. Vì đáp án có thể rất lớn, trả về nó **theo modulo `10^9 + 7`**.

**Ví dụ 1**

- Input: `n = 3`
- Output: `2`
- Giải thích: mảng ban đầu là `[1,2,3]`. Hai derangement là `[2,3,1]` và `[3,1,2]`.

**Ví dụ 2**

- Input: `n = 2`
- Output: `1`
- Giải thích: chỉ có `[2,1]`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `n = 1`
- Output: `0`
- Giải thích: mảng `[1]` chỉ có một hoán vị là chính nó, mà `1` đứng đúng vị trí nên không có derangement.

**Ràng buộc:**

- `1 <= n <= 10^6`

Chú ý: một hoán vị là derangement khi `perm[i] != i + 1` với **mọi** `i` (đánh chỉ số từ 0). Với `n = 10^6` không thể liệt kê hoán vị; phải đếm bằng công thức/quy hoạch động.

In [1]:
from typing import List
from itertools import permutations
from math import factorial, comb, e as EULER
from fractions import Fraction
import random
import sys
import time

MOD = 10**9 + 7

examples = [(3, 2), (2, 1), (1, 0)]
for n, expected in examples:
    print(f"n = {n} -> kỳ vọng {expected}")

n = 3 -> kỳ vọng 2
n = 2 -> kỳ vọng 1
n = 1 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Hoán vị và "điểm cố định"

Một **hoán vị** của `[1, ..., n]` là một cách sắp xếp lại `n` số đó. Phần tử ở **vị trí `i`** (đếm từ 1) mà đúng bằng `i` gọi là **điểm cố định** (fixed point). Derangement là hoán vị **không có điểm cố định**. `itertools.permutations(range(1, n + 1))` sinh ra mọi hoán vị.

In [2]:
for p in permutations([1, 2, 3]):
    fixed = [i + 1 for i, x in enumerate(p) if x == i + 1]
    print(p, "| điểm cố định:", fixed, "| derangement?" , len(fixed) == 0)

(1, 2, 3) | điểm cố định: [1, 2, 3] | derangement? False
(1, 3, 2) | điểm cố định: [1] | derangement? False
(2, 1, 3) | điểm cố định: [3] | derangement? False
(2, 3, 1) | điểm cố định: [] | derangement? True
(3, 1, 2) | điểm cố định: [] | derangement? True
(3, 2, 1) | điểm cố định: [2] | derangement? False


### 3.2. Số hoán vị là `n!` và tốc độ tăng của nó

`math.factorial(n)` tính `n! = 1 * 2 * ... * n`. Con số này tăng cực nhanh: `10! = 3.628.800`, `20! ≈ 2,4 * 10^18`, và `10^6!` có hàng triệu chữ số. Vì thế mọi cách "liệt kê hoán vị" chỉ chạy được với `n` rất nhỏ.

In [3]:
for n in (3, 5, 8, 10, 12, 15, 20):
    print(f"{n:>2}! = {factorial(n):,}")
print("số chữ số của 1000! =", len(str(factorial(1000))))

 3! = 6
 5! = 120
 8! = 40,320
10! = 3,628,800
12! = 479,001,600
15! = 1,307,674,368,000
20! = 2,432,902,008,176,640,000
số chữ số của 1000! = 2568


### 3.3. Phép modulo và các tính chất

Để tránh số khổng lồ, ta làm việc **theo modulo `M = 10^9 + 7`** ở **mọi bước**. Hai tính chất cần nhớ:

- `(a + b) % M == ((a % M) + (b % M)) % M`
- `(a * b) % M == ((a % M) * (b % M)) % M`

Vậy ta có thể lấy modulo **ngay sau mỗi phép cộng/nhân** mà kết quả cuối cùng không đổi. Trong Python, `%` với số dương `M` luôn cho kết quả **không âm** (kể cả khi vế trái âm), khác với C++/Java. Khoảng cách `10^9 + 7` là số nguyên tố, nên các giá trị luôn nằm trong `[0, M - 1]`.

In [4]:
a, b = 10**9 + 6, 5
print("(a + b) % M      =", (a + b) % MOD)
print("(a%M + b%M) % M  =", (a % MOD + b % MOD) % MOD)
print("(a * b) % M      =", (a * b) % MOD)
print("-3 % M           =", -3 % MOD, "(Python luôn cho kết quả không âm)")

(a + b) % M      = 4
(a%M + b%M) % M  = 4
(a * b) % M      = 1000000002
-3 % M           = 1000000004 (Python luôn cho kết quả không âm)


### 3.4. Gán nhiều biến đồng thời

`a, b = b, a + b` tính **vế phải trước** (dùng giá trị **cũ** của cả `a` và `b`) rồi mới gán. Rất tiện khi cập nhật hai biến "trượt" như trong dãy Fibonacci hay trong bài này.

In [5]:
a, b = 0, 1
for _ in range(6):
    a, b = b, a + b
    print(a, b)

# so với gán lần lượt (SAI nếu muốn dùng giá trị cũ)
a, b = 0, 1
a = b
b = a + b          # a đã bị đổi rồi!
print("gán lần lượt:", a, b, "(không phải 1, 1)")

1 1
1 2
2 3
3 5
5 8
8 13
gán lần lượt: 1 2 (không phải 1, 1)


### 3.5. Quy hoạch động và công thức truy hồi

**Quy hoạch động (DP)** giải bài lớn bằng cách giải các bài con nhỏ hơn **một lần** và nhớ kết quả. Với một **công thức truy hồi** (recurrence) như `f[i] = (i - 1) * (f[i - 1] + f[i - 2])`, ta chỉ cần biết hai giá trị trước để tính giá trị kế tiếp; ta dùng mảng hoặc chỉ hai biến.

### 3.6. Đệ quy và giới hạn độ sâu

Đệ quy gọi chính nó; Python mặc định giới hạn khoảng **1000 tầng** gọi hàm. Với `n = 10^6`, một hàm đệ quy `f(n) -> f(n - 1)` sẽ gây `RecursionError`. Vì vậy ta ưu tiên vòng `for` (lặp).

In [6]:
def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
print("depth(500) =", depth(500))
try:
    depth(10**6)
except RecursionError as ex:
    print("depth(10^6) -> RecursionError:", type(ex).__name__)

giới hạn đệ quy mặc định: 1000
depth(500) = 500
depth(10^6) -> RecursionError: RecursionError


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Gọi `D(n)` là số derangement của `[1..n]`.

**`n = 1`:** chỉ có `[1]`, có điểm cố định → `D(1) = 0`.

**`n = 2`:** hai hoán vị `[1,2]` (hai điểm cố định) và `[2,1]` (không có) → `D(2) = 1`.

**`n = 3`:** 6 hoán vị:

| hoán vị | điểm cố định | derangement? |
|---|---|---|
| `[1,2,3]` | 1, 2, 3 | không |
| `[1,3,2]` | 1 | không |
| `[2,1,3]` | 3 | không |
| `[2,3,1]` | không | **có** |
| `[3,1,2]` | không | **có** |
| `[3,2,1]` | 2 | không |

Vậy `D(3) = 2`.

**Tìm quy luật cho `n = 4`.** Hãy xem **số 1 được đặt ở vị trí nào**. Vì không được đặt ở vị trí 1, `1` có thể ở vị trí `2, 3` hoặc `4` — tức `n - 1 = 3` lựa chọn. Giả sử `1` ở vị trí `j`. Còn lại số `j` (số mà "vị trí gốc" của nó, vị trí `j`, đã bị `1` chiếm) phải đặt ở **một vị trí khác `j`**:

- Hoặc `j` đặt vào **vị trí 1** (hai số `1` và `j` hoán đổi chỗ cho nhau). Khi đó `n - 2` số còn lại phải tự derange → `D(n - 2)` cách.
- Hoặc `j` **không** đặt vào vị trí 1. Khi đó vị trí 1 đóng vai "vị trí cấm" của số `j` — một cấu trúc tương đương một bài derangement với `n - 1` phần tử → `D(n - 1)` cách.

Vậy: `D(n) = (n - 1) * (D(n - 1) + D(n - 2))`. Kiểm tra: `D(3) = 2 * (D(2) + D(1)) = 2 * (1 + 0) = 2` ✓, `D(4) = 3 * (2 + 1) = 9`.

In [7]:
def derangements_list(n):
    return [p for p in permutations(range(1, n + 1)) if all(x != i + 1 for i, x in enumerate(p))]

for n in range(1, 5):
    d = derangements_list(n)
    print(f"n = {n}: D = {len(d)}")
    if n <= 4:
        for p in d:
            print("   ", p)

D = {0: 1, 1: 0}
for i in range(2, 8):
    D[i] = (i - 1) * (D[i - 1] + D[i - 2])
print("theo công thức truy hồi:", D)

n = 1: D = 0
n = 2: D = 1
    (2, 1)
n = 3: D = 2
    (2, 3, 1)
    (3, 1, 2)
n = 4: D = 9
    (2, 1, 4, 3)
    (2, 3, 4, 1)
    (2, 4, 1, 3)
    (3, 1, 4, 2)
    (3, 4, 1, 2)
    (3, 4, 2, 1)
    (4, 1, 2, 3)
    (4, 3, 1, 2)
    (4, 3, 2, 1)
theo công thức truy hồi: {0: 1, 1: 0, 2: 1, 3: 2, 4: 9, 5: 44, 6: 265, 7: 1854}


## 5. Cách 1: Brute Force (liệt kê mọi hoán vị)

### Ý tưởng

Sinh **mọi** `n!` hoán vị bằng `itertools.permutations`, đếm các hoán vị không có điểm cố định (đúng theo định nghĩa). Chỉ chạy được với `n` nhỏ (khoảng `n <= 9`).

In [8]:
def find_derangement_brute(n):
    count = 0
    for p in permutations(range(1, n + 1)):
        if all(x != i + 1 for i, x in enumerate(p)):
            count += 1
    return count % MOD


for n, expected in examples:
    got = find_derangement_brute(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected
print([find_derangement_brute(n) for n in range(1, 9)])

3 -> 2 (kỳ vọng 2 )
2 -> 1 (kỳ vọng 1 )
1 -> 0 (kỳ vọng 0 )
[0, 1, 2, 9, 44, 265, 1854, 14833]


### Độ phức tạp

- Time `O(n * n!)` (có `n!` hoán vị, mỗi cái kiểm tra `O(n)`). Với `n = 10^6` thì vô nghĩa (số hoán vị có hàng triệu chữ số).
- Space `O(n)` cho một hoán vị đang xét.

### Nút thắt

Ta liệt kê cả những hoán vị **chắc chắn sai** (chẳng hạn bắt đầu bằng `1`) và ta **không cần biết hoán vị cụ thể là gì** — chỉ cần **số lượng**. Hãy đo sự bùng nổ:

In [9]:
for n in range(4, 10):
    t0 = time.perf_counter()
    r = find_derangement_brute(n)
    ms = (time.perf_counter() - t0) * 1000
    print(f"n = {n}: D = {r:>7} | số hoán vị phải duyệt = {factorial(n):>7,} | {ms:8.1f} ms")

n = 4: D =       9 | số hoán vị phải duyệt =      24 |      0.0 ms
n = 5: D =      44 | số hoán vị phải duyệt =     120 |      0.0 ms
n = 6: D =     265 | số hoán vị phải duyệt =     720 |      0.3 ms
n = 7: D =    1854 | số hoán vị phải duyệt =   5,040 |      2.2 ms
n = 8: D =   14833 | số hoán vị phải duyệt =  40,320 |     19.2 ms


n = 9: D =  133496 | số hoán vị phải duyệt = 362,880 |    181.0 ms


Thời gian nhân lên khoảng `n` lần mỗi khi `n` tăng 1 (đúng như `n!`). Với `n = 12` đã mất cả phút, `n = 10^6` là không thể. Cần một cách **đếm** mà không liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là nhận ra **cấu trúc đệ quy**: bài derangement cho `n` phần tử quy về hai bài **nhỏ hơn cùng dạng** (mục 4):

> `D(n) = (n - 1) * (D(n - 1) + D(n - 2))`, với `D(0) = 1`, `D(1) = 0`.

Đây là công thức **truy hồi hai số hạng**. Tính `D(0), D(1), ..., D(n)` lần lượt trong `O(n)` thời gian, mỗi bước lấy modulo.

Quy ước `D(0) = 1` (hoán vị rỗng tính là một derangement — "có đúng một cách không làm gì") là cần thiết để công thức đúng với `n = 2`: `D(2) = 1 * (D(1) + D(0)) = 1 * (0 + 1) = 1`.

Có thể đi đến cùng đáp án bằng nhiều con đường:

1. **Quay lui có cắt tỉa:** xây hoán vị từng vị trí, bỏ ngay nhánh vi phạm (vẫn là `O(n!)` nhưng nhanh hơn liệt kê).
2. **Bao hàm – loại trừ:** `D(n) = Σ (-1)^k * n! / k!`.
3. **DP hai số hạng:** `f[i] = (i - 1) * (f[i - 1] + f[i - 2])` (bảng, rồi tối ưu bộ nhớ thành hai biến).
4. **Công thức một số hạng:** `f[i] = i * f[i - 1] + (-1)^i`.

In [10]:
# minh họa: công thức truy hồi cho đúng giá trị brute với n <= 8
D = [1, 0]
for i in range(2, 9):
    D.append((i - 1) * (D[i - 1] + D[i - 2]))
for n in range(1, 9):
    assert D[n] == find_derangement_brute(n), n
print("D[0..8] =", D)

D[0..8] = [1, 0, 1, 2, 9, 44, 265, 1854, 14833]


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quay lui có cắt tỉa (backtracking)

**Ý tưởng khác:** xây hoán vị **từng vị trí một**. Ở vị trí `i` (từ 1), chọn một số **chưa dùng** và **khác `i`**. Nhánh nào vi phạm bị bỏ ngay, không sinh hoán vị đầy đủ rồi mới kiểm tra. Khi điền đủ `n` vị trí thì đếm 1.

**Giải thích từng đoạn:**

1. `used[v]` đánh dấu số `v` đã được đặt.
2. `place(i)`: nếu `i > n` thì đã điền xong → trả 1.
3. Với mỗi `v` trong `1..n` mà `v != i` và chưa dùng: đánh dấu, đếm đệ quy `place(i + 1)`, bỏ đánh dấu.

**Code demo:**

In [11]:
def backtrack_demo(n):
    used = [False] * (n + 1)
    perm = []
    found = []

    def place(i):
        if i > n:
            found.append(tuple(perm))
            return 1
        total = 0
        for v in range(1, n + 1):
            if v != i and not used[v]:
                used[v] = True
                perm.append(v)
                total += place(i + 1)
                perm.pop()
                used[v] = False
        return total

    return place(1), found

cnt, lst = backtrack_demo(4)
print(cnt, lst)

9 [(2, 1, 4, 3), (2, 3, 4, 1), (2, 4, 1, 3), (3, 1, 4, 2), (3, 4, 1, 2), (3, 4, 2, 1), (4, 1, 2, 3), (4, 3, 1, 2), (4, 3, 2, 1)]


**Code nộp LeetCode (cách 2, chỉ đúng với `n` nhỏ — không qua được `n = 10^6`; để so sánh):**

In [12]:
class Solution2:
    def findDerangement(self, n: int) -> int:
        used = [False] * (n + 1)

        def place(i):
            if i > n:
                return 1
            total = 0
            for v in range(1, n + 1):
                if v != i and not used[v]:
                    used[v] = True
                    total += place(i + 1)
                    used[v] = False
            return total

        return place(1) % (10**9 + 7)


for n, expected in examples:
    assert Solution2().findDerangement(n) == expected
print("Cách 2 OK (n nhỏ)")

Cách 2 OK (n nhỏ)


**Độ phức tạp:** Time `O(n!)` trong trường hợp xấu (số nhánh còn lại cỡ `D(n)` nhân hằng số nhỏ; vẫn mũ giai thừa). Nhanh hơn cách 1 khoảng vài lần nhờ cắt sớm nhưng cùng bậc. Space `O(n)` (đệ quy sâu `n` + `used`).

### 7.2. Cách 3 — Bao hàm – loại trừ (inclusion–exclusion)

**Ý tưởng khác:** đếm **gián tiếp**. Tổng số hoán vị là `n!`. Trừ đi các hoán vị có **ít nhất một** điểm cố định. Dùng nguyên lý bao hàm – loại trừ: gọi `A_i` là tập hoán vị có `i` là điểm cố định. Với mỗi cách chọn `k` điểm cố định, số hoán vị tương ứng là `(n - k)!`; có `C(n, k)` cách chọn. Vậy

`D(n) = Σ_{k = 0}^{n} (-1)^k * C(n, k) * (n - k)! = Σ_{k = 0}^{n} (-1)^k * n! / k!`

**Giải thích từng đoạn:**

1. Tổng đan dấu `+n!/0! - n!/1! + n!/2! - ...`.
2. Số `n! / k!` bằng tích `(k + 1)(k + 2)...n`. Duyệt `k` từ `n` xuống `0` thì `n!/k!` được cập nhật bằng **một phép nhân**: `term_{k-1} = term_k * k`. Như vậy toàn bộ tổng tính trong `O(n)` phép nhân, **không cần phép chia modulo**.
3. Lấy modulo sau mỗi bước.

**Code demo (với số nguyên chính xác, n nhỏ):**

In [13]:
def inclusion_exclusion_exact(n):
    return sum((-1) ** k * factorial(n) // factorial(k) for k in range(n + 1))

for n in range(0, 9):
    print(f"D({n}) = {inclusion_exclusion_exact(n)}")

# chi tiết cho n = 4
n = 4
terms = [(-1) ** k * factorial(n) // factorial(k) for k in range(n + 1)]
print("các số hạng n=4:", terms, "-> tổng =", sum(terms))

D(0) = 1
D(1) = 0
D(2) = 1
D(3) = 2
D(4) = 9
D(5) = 44
D(6) = 265
D(7) = 1854
D(8) = 14833
các số hạng n=4: [24, -24, 12, -4, 1] -> tổng = 9


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def findDerangement(self, n: int) -> int:
        mod = 10**9 + 7
        term = 1                      # n! / n! = 1 (ứng với k = n)
        total = 0
        for k in range(n, -1, -1):    # k = n, n-1, ..., 0
            sign = 1 if k % 2 == 0 else -1
            total = (total + sign * term) % mod
            term = term * k % mod     # chuyển sang n!/(k-1)! = n!/k! * k
        return total % mod


for n, expected in examples:
    assert Solution3().findDerangement(n) == expected
print("Cách 3 OK")

Cách 3 OK


Khi `k = 0` thì `term * 0` bằng 0 nhưng đó là bước cuối, sau đó vòng lặp kết thúc nên không ảnh hưởng.

**Độ phức tạp:** Time `O(n)`; Space `O(1)`.

### 7.3. Cách 4 — Quy hoạch động hai số hạng (bảng, rồi tối ưu bộ nhớ)

**Ý tưởng khác:** dùng ngay công thức truy hồi ở mục 4: `f[i] = (i - 1) * (f[i - 1] + f[i - 2])`, `f[0] = 1`, `f[1] = 0`.

**Giải thích từng đoạn:**

1. Trường hợp cơ sở: `f[0] = 1`, `f[1] = 0`.
2. Với `i` từ `2` đến `n`: `f[i] = (i - 1) * (f[i - 1] + f[i - 2]) % MOD`.
3. Đáp án `f[n]`.
4. Tối ưu bộ nhớ: mỗi lần chỉ dùng hai giá trị trước, nên chỉ cần hai biến `a = f[i - 2]`, `b = f[i - 1]` và cập nhật bằng `a, b = b, (i - 1) * (a + b) % MOD`.

**Code demo (bảng đầy đủ, in từng bước):**

In [15]:
def dp_table_demo(n):
    f = [0] * (n + 1)
    f[0] = 1
    if n >= 1:
        f[1] = 0
    print("f[0] =", f[0], "| f[1] =", f[1] if n >= 1 else None)
    for i in range(2, n + 1):
        f[i] = (i - 1) * (f[i - 1] + f[i - 2]) % MOD
        print(f"f[{i}] = ({i} - 1) * (f[{i - 1}] + f[{i - 2}]) = {i - 1} * ({f[i - 1]} + {f[i - 2]}) = {f[i]}")
    return f[n]

print("đáp án:", dp_table_demo(6))

f[0] = 1 | f[1] = 0
f[2] = (2 - 1) * (f[1] + f[0]) = 1 * (0 + 1) = 1
f[3] = (3 - 1) * (f[2] + f[1]) = 2 * (1 + 0) = 2
f[4] = (4 - 1) * (f[3] + f[2]) = 3 * (2 + 1) = 9
f[5] = (5 - 1) * (f[4] + f[3]) = 4 * (9 + 2) = 44
f[6] = (6 - 1) * (f[5] + f[4]) = 5 * (44 + 9) = 265
đáp án: 265


**Code nộp LeetCode (cách 4, bản hai biến — O(1) bộ nhớ):**

In [16]:
class Solution4:
    def findDerangement(self, n: int) -> int:
        mod = 10**9 + 7
        a, b = 1, 0                    # a = f[0], b = f[1]
        for i in range(2, n + 1):
            a, b = b, (i - 1) * (a + b) % mod
        return b if n >= 1 else a


for n, expected in examples:
    assert Solution4().findDerangement(n) == expected
print("Cách 4 OK")

Cách 4 OK


Với `n = 1` vòng lặp không chạy và trả `b = 0` ✓. Dòng `a, b = b, ...` tính vế phải với giá trị **cũ** của `a, b`, rồi mới gán — đúng như ta cần (xem mục 3.4).

**Độ phức tạp:** Time `O(n)`; Space `O(1)` (bản bảng là `O(n)`).

### 7.4. Cách 5 — Công thức một số hạng `f[i] = i * f[i - 1] + (-1)^i`

**Ý tưởng khác:** từ cùng công thức hai số hạng ta rút ra một công thức **chỉ phụ thuộc `f[i - 1]`**:

`D(i) = i * D(i - 1) + (-1)^i`, với `D(0) = 1`.

Kiểm tra: `D(1) = 1 * 1 - 1 = 0`; `D(2) = 2 * 0 + 1 = 1`; `D(3) = 3 * 1 - 1 = 2`; `D(4) = 4 * 2 + 1 = 9`; `D(5) = 5 * 9 - 1 = 44` ✓.

**Giải thích từng đoạn:** Ta chỉ cần **một** biến `f`, khởi tạo `f = 1` (ứng với `D(0)`). Với `i` từ `1` đến `n`: `f = (i * f + (1 if i % 2 == 0 else -1)) % MOD`.

Vì sao đúng? Từ `D(i) = (i - 1)(D(i - 1) + D(i - 2))` ta chứng minh quy nạp rằng `D(i) - i * D(i - 1) = (-1)^i`. Nếu đặt `g(i) = D(i) - i * D(i - 1)` thì `g(i) = (i - 1) D(i - 1) + (i - 1) D(i - 2) - i D(i - 1) = (i - 1) D(i - 2) - D(i - 1) = -(D(i - 1) - (i - 1) D(i - 2)) = -g(i - 1)`. Mà `g(1) = D(1) - 1 * D(0) = 0 - 1 = -1`, nên `g(i) = (-1)^i`.

**Code demo:**

In [17]:
def one_term_demo(n):
    f = 1
    print("f(0) = 1")
    for i in range(1, n + 1):
        sign = 1 if i % 2 == 0 else -1
        f = i * f + sign
        print(f"f({i}) = {i} * f({i - 1}) + ({sign:+d}) = {f}")
    return f

print("đáp án:", one_term_demo(7))

f(0) = 1
f(1) = 1 * f(0) + (-1) = 0
f(2) = 2 * f(1) + (+1) = 1
f(3) = 3 * f(2) + (-1) = 2
f(4) = 4 * f(3) + (+1) = 9
f(5) = 5 * f(4) + (-1) = 44
f(6) = 6 * f(5) + (+1) = 265
f(7) = 7 * f(6) + (-1) = 1854
đáp án: 1854


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def findDerangement(self, n: int) -> int:
        mod = 10**9 + 7
        f = 1                                  # D(0)
        for i in range(1, n + 1):
            f = (i * f + (1 if i % 2 == 0 else -1)) % mod
        return f


for n, expected in examples:
    assert Solution5().findDerangement(n) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** Time `O(n)`; Space `O(1)`. Ngắn nhất và chỉ dùng **một** biến, nhưng khó nhìn ra ý nghĩa tổ hợp hơn cách 4. (Trong Python `% mod` luôn cho kết quả không âm. Ở C++/Java, nếu biểu thức có thể âm thì phải viết `((x % mod) + mod) % mod`.)

## 8. TRACE từng bước

### 8.1. Trace cách 4 (DP hai số hạng) với `n = 6`

Bảng trạng thái (mỗi dòng ứng với một giá trị `i`):

| Cột | Ý nghĩa |
|---|---|
| `i` | độ dài hiện tại |
| `a = f[i-2]`, `b = f[i-1]` | hai giá trị trước đó |
| `(i-1)*(a+b)` | giá trị mới `f[i]` (trước modulo) |

In [19]:
def trace_dp(n):
    a, b = 1, 0
    print(f"khởi tạo: a = f[0] = {a}, b = f[1] = {b}")
    for i in range(2, n + 1):
        new = (i - 1) * (a + b)
        print(f"i = {i}: a = {a:>3}, b = {b:>3} | ({i} - 1) * ({a} + {b}) = {new:>4} -> f[{i}] = {new % MOD}")
        a, b = b, new % MOD
    return b

print("kết quả:", trace_dp(6))

khởi tạo: a = f[0] = 1, b = f[1] = 0
i = 2: a =   1, b =   0 | (2 - 1) * (1 + 0) =    1 -> f[2] = 1
i = 3: a =   0, b =   1 | (3 - 1) * (0 + 1) =    2 -> f[3] = 2
i = 4: a =   1, b =   2 | (4 - 1) * (1 + 2) =    9 -> f[4] = 9
i = 5: a =   2, b =   9 | (5 - 1) * (2 + 9) =   44 -> f[5] = 44
i = 6: a =   9, b =  44 | (6 - 1) * (9 + 44) =  265 -> f[6] = 265
kết quả: 265


Quan sát: dãy `1, 0, 1, 2, 9, 44, 265` — mỗi số mới là `(i - 1)` lần tổng hai số trước. Đây là dãy số "subfactorial" `!n` quen thuộc.

### 8.2. Trace ý nghĩa tổ hợp của `D(4) = 3 * (D(3) + D(2)) = 9`

Ta liệt kê **9 derangement của `[1,2,3,4]`**, nhóm theo vị trí `j` của số `1` và xem số `j` có quay về vị trí 1 (hoán đổi) hay không. Mỗi nhóm `j` có `D(3) + D(2) = 2 + 1 = 3` hoán vị: `1` nhánh hoán đổi (`D(2) = 1`) và `2` nhánh không hoán đổi (`D(3) = 2`).

In [20]:
def trace_structure(n):
    ders = derangements_list(n)
    print(f"n = {n}: có {len(ders)} derangement")
    for j in range(2, n + 1):
        group = [p for p in ders if p.index(1) + 1 == j]       # số 1 nằm ở vị trí j
        swap = [p for p in group if p[0] == j]                 # số j quay về vị trí 1
        other = [p for p in group if p[0] != j]
        print(f"  số 1 ở vị trí {j}: {len(group)} hoán vị = hoán đổi {len(swap)} + không hoán đổi {len(other)}")
        print(f"      hoán đổi (còn {n - 2} số tự derange): {swap}")
        print(f"      không hoán đổi:                      {other}")

trace_structure(4)

n = 4: có 9 derangement
  số 1 ở vị trí 2: 3 hoán vị = hoán đổi 1 + không hoán đổi 2
      hoán đổi (còn 2 số tự derange): [(2, 1, 4, 3)]
      không hoán đổi:                      [(3, 1, 4, 2), (4, 1, 2, 3)]
  số 1 ở vị trí 3: 3 hoán vị = hoán đổi 1 + không hoán đổi 2
      hoán đổi (còn 2 số tự derange): [(3, 4, 1, 2)]
      không hoán đổi:                      [(2, 4, 1, 3), (4, 3, 1, 2)]
  số 1 ở vị trí 4: 3 hoán vị = hoán đổi 1 + không hoán đổi 2
      hoán đổi (còn 2 số tự derange): [(4, 3, 2, 1)]
      không hoán đổi:                      [(2, 3, 4, 1), (3, 4, 2, 1)]


Mỗi vị trí `j ∈ {2, 3, 4}` cho `1 + 2 = 3` hoán vị: đúng `D(2) + D(3)`; có `3` vị trí nên `3 * 3 = 9`.

### 8.3. Trace cách 3 (bao hàm – loại trừ) với `n = 5`

In [21]:
def trace_inclusion_exclusion(n):
    term, total = 1, 0
    for k in range(n, -1, -1):
        sign = 1 if k % 2 == 0 else -1
        total += sign * term
        print(f"k = {k}: n!/k! = {term:>4} | dấu {sign:+d} | tổng tạm = {total}")
        term *= k
    return total

print("kết quả D(5) =", trace_inclusion_exclusion(5))

k = 5: n!/k! =    1 | dấu -1 | tổng tạm = -1
k = 4: n!/k! =    5 | dấu +1 | tổng tạm = 4
k = 3: n!/k! =   20 | dấu -1 | tổng tạm = -16
k = 2: n!/k! =   60 | dấu +1 | tổng tạm = 44
k = 1: n!/k! =  120 | dấu -1 | tổng tạm = -76
k = 0: n!/k! =  120 | dấu +1 | tổng tạm = 44
kết quả D(5) = 44


Các số hạng `120, -120, +60, -20, +5, -1` cộng lại `44` — hiệu của `5! = 120` trừ những hoán vị có điểm cố định (bao hàm – loại trừ điều chỉnh số lần đếm trùng).

### 8.4. Trace cách 5 với `n = 6`

In [22]:
def trace_one_term(n):
    f = 1
    print("f(0) = 1")
    for i in range(1, n + 1):
        sign = 1 if i % 2 == 0 else -1
        old = f
        f = i * f + sign
        print(f"i = {i}: {i} * {old} {sign:+d} = {f}")
    return f

print("kết quả:", trace_one_term(6))

f(0) = 1
i = 1: 1 * 1 -1 = 0
i = 2: 2 * 0 +1 = 1
i = 3: 3 * 1 -1 = 2
i = 4: 4 * 2 +1 = 9
i = 5: 5 * 9 -1 = 44
i = 6: 6 * 44 +1 = 265
kết quả: 265


## 9. Code nộp LeetCode

Chọn **cách 4 (DP hai số hạng, hai biến)**: có ý nghĩa tổ hợp rõ ràng, `O(n)` thời gian, `O(1)` bộ nhớ, vòng lặp (không đệ quy) nên chịu được `n = 10^6`. Đây là code đúng class/method/signature:

In [23]:
class Solution:
    def findDerangement(self, n: int) -> int:
        mod = 10**9 + 7
        a, b = 1, 0                    # a = f[0], b = f[1]
        for i in range(2, n + 1):
            a, b = b, (i - 1) * (a + b) % mod
        return b if n >= 1 else a


for n, expected in examples:
    got = Solution().findDerangement(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected
print("n = 10^6 ->", Solution().findDerangement(10**6))

3 -> 2 (kỳ vọng 2 )
2 -> 1 (kỳ vọng 1 )
1 -> 0 (kỳ vọng 0 )


n = 10^6 -> 102701088


## 10. Vì sao thuật toán đúng

**Công thức truy hồi.** Gọi `D(n)` là số derangement của `[1..n]` (hoán vị `p` với `p[i] != i` mọi `i`). Xét `n >= 2` và vị trí `j = p^{-1}(1)` nơi số `1` đang đứng; `j != 1` vì `p[1] != 1`, nên có `n - 1` cách chọn `j`. Với mỗi `j`, chia hai trường hợp theo giá trị `p[1]` (số đang đứng ở vị trí 1):

- **`p[1] = j`** (số `j` về vị trí 1, `1` ở vị trí `j`): hai vị trí `1, j` và hai số `1, j` đã được "ghép đôi cho nhau" hợp lệ (`1` không ở vị trí 1, `j` không ở vị trí `j`). Còn lại `n - 2` số trên `n - 2` vị trí; mỗi số phải khác vị trí gốc → bài derangement cho `n - 2` phần tử. Có `D(n - 2)` cách.
- **`p[1] != j`:** ta "gộp" `1` vào vị trí `j`: coi vị trí `j` như bị số `1` chiếm, và **nhãn của vị trí 1** thay cho nhãn của vị trí `j`. Cụ thể, ánh xạ: bỏ số `1` và vị trí `j`, rồi đổi tên vị trí `1` thành `j`. Khi đó mỗi số còn lại trong `{2..n}` có đúng một vị trí cấm — số `j` giờ có vị trí cấm là vị trí (đã đổi tên) `j`, tức vị trí `1` thật; các số khác có vị trí cấm là vị trí gốc của chúng. Đây là một song ánh với derangement của `n - 1` phần tử → `D(n - 1)` cách.

Hai trường hợp rời nhau và đủ, nên `D(n) = (n - 1)(D(n - 1) + D(n - 2))`. Trường hợp cơ sở: `D(0) = 1` (hoán vị rỗng), `D(1) = 0`. Công thức đúng với `n = 2` (`1 * (0 + 1) = 1`).

**Cách 4.** Quy nạp trên `i` cho thấy sau lần lặp thứ `i`, `(a, b) = (D(i - 1), D(i))`; modulo bảo toàn vì phép cộng và nhân tương thích với modulo (mục 3.3).

**Cách 5.** Chứng minh ở mục 7.4: `D(i) - i D(i - 1) = (-1)^i` bằng quy nạp từ công thức truy hồi.

**Cách 3.** Nguyên lý bao hàm – loại trừ: số hoán vị không có điểm cố định nào `= Σ_{S ⊆ [n]} (-1)^{|S|} * (số hoán vị cố định mọi phần tử của S)` `= Σ_k (-1)^k C(n, k) (n - k)!` vì cố định `k` phần tử thì `n - k` phần tử còn lại hoán vị tự do. `C(n, k) (n - k)! = n! / k!`.

**Cách 1, 2.** Là bản dịch trực tiếp của định nghĩa.

Cả năm cách đều trả cùng một dãy `1, 0, 1, 2, 9, 44, 265, 1854, ...` — ta kiểm chứng bằng code ở mục 12.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi hoán vị | `O(n * n!)` | `O(n)` |
| 2. Quay lui có cắt tỉa | `O(n!)` | `O(n)` |
| 3. Bao hàm – loại trừ | `O(n)` | `O(1)` |
| 4. DP hai số hạng (nộp) | `O(n)` (bảng: `O(n)` bộ nhớ, hai biến: `O(1)`) | `O(1)` |
| 5. Công thức một số hạng | `O(n)` | `O(1)` |

Với `n = 10^6`, ba cách tuyến tính chỉ cần cỡ `10^6` phép nhân modulo — dưới một giây trong Python. Đo thực tế:

In [24]:
for n in (10**4, 10**5, 10**6):
    row = []
    answers = set()
    for label, f in (("cách 3", Solution3().findDerangement), ("cách 4", Solution4().findDerangement),
                     ("cách 5", Solution5().findDerangement)):
        t0 = time.perf_counter()
        r = f(n)
        ms = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{label} {ms:7.1f} ms")
    assert len(answers) == 1
    print(f"n = {n:>8}: D(n) mod p = {answers.pop():>10} | " + " | ".join(row))

n =    10000: D(n) mod p =  381587473 | cách 3     1.3 ms | cách 4     0.9 ms | cách 5     0.9 ms


n =   100000: D(n) mod p =  183389504 | cách 3    14.2 ms | cách 4    10.5 ms | cách 5    10.2 ms


n =  1000000: D(n) mod p =  102701088 | cách 3   156.9 ms | cách 4   122.1 ms | cách 5    91.4 ms


Cả ba cách đều chạy tốt với `n = 10^6`.

## 12. Bộ test

### 12.1. Ví dụ của đề, dãy số đã biết và edge case

Dãy derangement (OEIS A000166) bắt đầu: `1, 0, 1, 2, 9, 44, 265, 1854, 14833, 133496, 1334961, 14684570, 176214841, ...`. Ta dùng nó như một **oracle độc lập** (đã biết từ trước, không do code của ta sinh ra), bên cạnh vét cạn.

In [25]:
KNOWN = [1, 0, 1, 2, 9, 44, 265, 1854, 14833, 133496, 1334961, 14684570, 176214841,
         2290792932, 32071101049, 481066515734]          # D(0) .. D(15)

solvers = [
    ("cách 2", Solution2().findDerangement),
    ("cách 3", Solution3().findDerangement),
    ("cách 4", Solution4().findDerangement),
    ("cách 5", Solution5().findDerangement),
]


def check_small(n):
    expected = KNOWN[n] % MOD
    for name, f in solvers:
        if name == "cách 2" and n > 9:
            continue                      # quay lui chỉ chạy được với n nhỏ
        got = f(n)
        assert got == expected, (name, n, got, expected)
    if n <= 8:
        assert find_derangement_brute(n) == expected, n
    return expected


for n, expected in examples:
    assert check_small(n) == expected
print("ví dụ đề: OK")

for n in range(1, 16):
    check_small(n)
print("n = 1..15 khớp dãy đã biết (n <= 8 còn khớp vét cạn, n <= 9 khớp quay lui): OK")
print("D(n) mod p với n = 1..15:", [KNOWN[n] % MOD for n in range(1, 16)])

ví dụ đề: OK
n = 1..15 khớp dãy đã biết (n <= 8 còn khớp vét cạn, n <= 9 khớp quay lui): OK
D(n) mod p với n = 1..15: [0, 1, 2, 9, 44, 265, 1854, 14833, 133496, 1334961, 14684570, 176214841, 290792918, 71100825, 66512367]


Lưu ý `D(13) = 2290792932 > 10^9 + 7`, nên từ `n = 13` đáp án **bị rút gọn theo modulo** (`2290792932 % (10^9+7) = 290792918`), một nơi dễ sai nếu quên `%`.

### 12.2. Phản ví dụ cho các suy nghĩ sai

Ba "công thức" nghe có lý nhưng sai:

In [26]:
def wrong_factorial_minus(n):
    """SAI: n! - n * (n-1)! ... (tưởng trừ các hoán vị có điểm cố định là xong)."""
    return factorial(n) - n * factorial(n - 1)


def wrong_n_minus_1_factorial(n):
    """SAI: (n - 1)! ... tưởng cố định 1 rồi hoán vị phần còn lại."""
    return factorial(n - 1)


def wrong_recurrence_n(n):
    """SAI: nhân n thay vì (n - 1) trong công thức truy hồi."""
    a, b = 1, 0
    for i in range(2, n + 1):
        a, b = b, i * (a + b)
    return b if n >= 1 else a


print(f"{'n':>2} | {'đúng':>6} | {'n!-n(n-1)!':>10} | {'(n-1)!':>6} | {'nhân n':>6}")
for n in range(1, 8):
    print(f"{n:>2} | {KNOWN[n]:>6} | {wrong_factorial_minus(n):>10} | {wrong_n_minus_1_factorial(n):>6} | {wrong_recurrence_n(n):>6}")

assert wrong_factorial_minus(3) != KNOWN[3]
assert wrong_n_minus_1_factorial(4) != KNOWN[4]      # n = 3 trùng ngẫu nhiên (2! = 2)
assert wrong_recurrence_n(4) != KNOWN[4]

 n |   đúng | n!-n(n-1)! | (n-1)! | nhân n
 1 |      0 |          0 |      1 |      0
 2 |      1 |          0 |      1 |      2
 3 |      2 |          0 |      2 |      6
 4 |      9 |          0 |      6 |     32
 5 |     44 |          0 |     24 |    190
 6 |    265 |          0 |    120 |   1332
 7 |   1854 |          0 |    720 |  10654


- `n! - n * (n - 1)!` luôn bằng `0`: vì `n * (n - 1)! = n!`. Ý đồ "trừ hoán vị có điểm cố định" thất bại do **đếm trùng**: hoán vị có hai điểm cố định bị trừ nhiều lần; bao hàm – loại trừ phải **cộng lại** chúng (dấu luân phiên).
- `(n - 1)!` đếm hoán vị mà `1` đứng cố định, không phải lệch hết; nó trùng `D(3) = 2` chỉ do ngẫu nhiên, sai từ `n = 4` (`3! = 6` còn `D(4) = 9`).
- Nhân `i` thay `(i - 1)` cho giá trị lệch hẳn từ `n = 3` trở đi.

### 12.3. Đối chiếu nhiều `n` (toàn bộ `n` nhỏ + ngẫu nhiên có seed)

Trước hết, bằng số nguyên chính xác (không modulo), ta kiểm tra **công thức truy hồi và công thức một số hạng cho ra cùng dãy** đến `n = 600`, và đối chiếu với `Σ (-1)^k n!/k!` chính xác. Sau đó so các cách modulo cho `n` ngẫu nhiên đến `10^6`.

In [27]:
# 1) so sánh chính xác (số nguyên lớn) đến n = 600
exact_two = [1, 0]
for i in range(2, 601):
    exact_two.append((i - 1) * (exact_two[i - 1] + exact_two[i - 2]))
exact_one = [1]
for i in range(1, 601):
    exact_one.append(i * exact_one[i - 1] + (-1) ** i)
assert exact_two == exact_one
for n in (0, 1, 2, 5, 10, 50, 200, 600):
    assert exact_two[n] == inclusion_exclusion_exact(n), n
assert exact_two[:16] == KNOWN
print("n = 0..600: truy hồi hai số hạng = một số hạng; các giá trị chọn lọc khớp bao hàm – loại trừ chính xác")

# 2) mọi n nhỏ: cách nộp khớp D(n) % MOD tính chính xác
for n in range(1, 601):
    assert Solution().findDerangement(n) == exact_two[n] % MOD, n
print("n = 1..600: cách nộp khớp đáp án chính xác % MOD")

# 3) xấp xỉ D(n) ~ n!/e cho n <= 15 (kiểm tra độc lập thứ ba)
for n in range(1, 16):
    assert round(factorial(n) / EULER) == KNOWN[n], n
print("n = 1..15: D(n) = round(n!/e) (nhận xét toán học)")

n = 0..600: truy hồi hai số hạng = một số hạng; các giá trị chọn lọc khớp bao hàm – loại trừ chính xác
n = 1..600: cách nộp khớp đáp án chính xác % MOD
n = 1..15: D(n) = round(n!/e) (nhận xét toán học)


In [28]:
rng = random.Random(20634)
ns = [rng.randint(1, 10**6) for _ in range(12)] + [10**6, 10**6 - 1, 999_999, 500_000]
for n in ns:
    r3 = Solution3().findDerangement(n)
    r4 = Solution4().findDerangement(n)
    r5 = Solution5().findDerangement(n)
    assert r3 == r4 == r5 == Solution().findDerangement(n), n
print(f"{len(ns)} giá trị n ngẫu nhiên và cực biên (tới 10^6): cách 3, 4, 5 và cách nộp trùng nhau")

# kiểm tra tính chất: D(n) = n*D(n-1) + (-1)^n  (mod p) với n ngẫu nhiên lớn
for n in rng.sample(range(2, 10**6), 6):
    lhs = Solution().findDerangement(n)
    rhs = (n * Solution().findDerangement(n - 1) + (-1) ** n) % MOD
    assert lhs == rhs, n
print("tính chất D(n) = n*D(n-1) + (-1)^n đúng với các n lớn ngẫu nhiên")

16 giá trị n ngẫu nhiên và cực biên (tới 10^6): cách 3, 4, 5 và cách nộp trùng nhau


tính chất D(n) = n*D(n-1) + (-1)^n đúng với các n lớn ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên lấy modulo
Python không tràn số, nên chương trình vẫn **chạy** nhưng cho đáp án **không theo modulo** (và chậm vì số lớn).

**Code sai:**

In [29]:
class Wrong1:
    def findDerangement(self, n):
        a, b = 1, 0
        for i in range(2, n + 1):
            a, b = b, (i - 1) * (a + b)        # SAI: không % mod
        return b if n >= 1 else a

for n in (10, 13, 20):
    print(f"n = {n}: sai = {Wrong1().findDerangement(n)} | đúng = {Solution().findDerangement(n)}")

t0 = time.perf_counter()
Wrong1().findDerangement(20000)
t1 = time.perf_counter() - t0
t0 = time.perf_counter()
Solution().findDerangement(20000)
t2 = time.perf_counter() - t0
print(f"n = 20000: không modulo {t1 * 1000:8.1f} ms | có modulo {t2 * 1000:6.1f} ms (số khổng lồ làm chậm)")

n = 10: sai = 1334961 | đúng = 1334961
n = 13: sai = 2290792932 | đúng = 290792918
n = 20: sai = 895014631192902121 | đúng = 927799753
n = 20000: không modulo    184.9 ms | có modulo    1.9 ms (số khổng lồ làm chậm)


**Vì sao sai:** đề yêu cầu kết quả theo modulo `10^9 + 7`; từ `n = 13` giá trị đã vượt `10^9 + 7`. Hơn nữa không modulo thì số có tới hàng chục nghìn chữ số, nhân/cộng chậm dần và với `n = 10^6` không chạy nổi. **Luôn `% mod` sau mỗi phép nhân/cộng.**

### Lỗi 2: Sai trường hợp cơ sở

In [30]:
def wrong_base(n, f0, f1):
    a, b = f0, f1
    for i in range(2, n + 1):
        a, b = b, (i - 1) * (a + b) % MOD
    return b if n >= 1 else a

for (f0, f1) in ((0, 0), (1, 1), (0, 1), (1, 0)):
    print(f"f0 = {f0}, f1 = {f1}:", [wrong_base(n, f0, f1) for n in range(1, 8)])
print("đúng (1, 0)  :", KNOWN[1:8])

f0 = 0, f1 = 0: [0, 0, 0, 0, 0, 0, 0]
f0 = 1, f1 = 1: [1, 2, 6, 24, 120, 720, 5040]
f0 = 0, f1 = 1: [1, 1, 4, 15, 76, 455, 3186]
f0 = 1, f1 = 0: [0, 1, 2, 9, 44, 265, 1854]
đúng (1, 0)  : [0, 1, 2, 9, 44, 265, 1854]


**Vì sao sai:** chỉ cặp `f[0] = 1`, `f[1] = 0` tạo ra dãy đúng `0, 1, 2, 9, 44, ...`. Nếu `f[0] = 0` thì mọi giá trị đều `0`; nếu `f[1] = 1` thì thành dãy khác. Cái dễ nhầm nhất là `f[0]`: "0 phần tử có mấy derangement?" — **đúng 1** (hoán vị rỗng).

### Lỗi 3: Nhân `i` thay vì `i - 1`
Đã minh họa ở 12.2 (`wrong_recurrence_n`). Hệ số `(i - 1)` chính là số vị trí mà số `1` có thể đến (mọi vị trí trừ vị trí 1).

In [31]:
print("nhân i      :", [wrong_recurrence_n(n) for n in range(1, 8)])
print("nhân (i - 1):", KNOWN[1:8])

nhân i      : [0, 2, 6, 32, 190, 1332, 10654]
nhân (i - 1): [0, 1, 2, 9, 44, 265, 1854]


### Lỗi 4: Đệ quy cho `n` lớn — `RecursionError`

**Code sai:**

In [32]:
def derange_recursive(n):
    if n == 0:
        return 1
    if n == 1:
        return 0
    return (n - 1) * (derange_recursive(n - 1) + derange_recursive(n - 2)) % MOD   # cũng nổ tung số lần gọi

print("n = 15 (nhỏ) :", derange_recursive(15))
try:
    derange_recursive(2000)
except RecursionError:
    print("n = 2000 -> RecursionError (vượt giới hạn tầng đệ quy)")
print("đúng với n = 10^6:", Solution().findDerangement(10**6))

n = 15 (nhỏ) : 66512367
n = 2000 -> RecursionError (vượt giới hạn tầng đệ quy)


đúng với n = 10^6: 102701088


**Vì sao sai:** đệ quy thuần túy có hai vấn đề: (1) **độ sâu** `n` vượt giới hạn tầng (mặc định ~1000), (2) không nhớ kết quả nên số lần gọi tăng **mũ** (như Fibonacci ngây thơ). Nếu muốn đệ quy thì phải thêm memo **và** nâng giới hạn — dễ lỗi; dùng vòng `for` bottom-up là an toàn nhất.

### Lỗi 5: Cập nhật hai biến sai thứ tự

In [33]:
class Wrong5:
    def findDerangement(self, n):
        a, b = 1, 0
        for i in range(2, n + 1):
            a = b                              # SAI: gán lần lượt, b mới tính theo a đã bị đổi
            b = (i - 1) * (a + b) % MOD
        return b if n >= 1 else a

print("sai :", [Wrong5().findDerangement(n) for n in range(1, 8)])
print("đúng:", [Solution().findDerangement(n) for n in range(1, 8)])

sai : [0, 0, 0, 0, 0, 0, 0]
đúng: [0, 1, 2, 9, 44, 265, 1854]


**Vì sao sai:** sau `a = b`, biến `a` đã mất giá trị cũ `f[i - 2]` nên dòng sau dùng `a + b = 2 * b`. Dùng gán đồng thời `a, b = b, (i - 1) * (a + b) % mod` (vế phải tính hết bằng giá trị cũ rồi mới gán).

### Lỗi 6: Dùng số thực `n! / e` cho `n` lớn

In [34]:
def wrong_float(n):
    return round(factorial(n) / EULER)

for n in (10, 15, 18, 20, 22):
    exact = exact_two[n]
    approx = wrong_float(n)
    print(f"n = {n:>2}: đúng = {exact:>22} | round(n!/e) = {approx:>22} | {'khớp' if approx == exact else 'LỆCH'}")

n = 10: đúng =                1334961 | round(n!/e) =                1334961 | khớp
n = 15: đúng =           481066515734 | round(n!/e) =           481066515734 | khớp
n = 18: đúng =       2355301661033953 | round(n!/e) =       2355301661033953 | khớp
n = 20: đúng =     895014631192902121 | round(n!/e) =     895014631192902144 | LỆCH
n = 22: đúng =  413496759611120779881 | round(n!/e) =  413496759611120812032 | LỆCH


**Vì sao sai:** `D(n) = round(n! / e)` là đúng về toán học nhưng `float` chỉ có ~15–16 chữ số có nghĩa; từ `n = 20` kết quả đã lệch. Hơn nữa với `n = 10^6` thì `n!` không thể tính. Với bài chính xác/modulo hãy dùng số nguyên và công thức truy hồi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm số hoán vị/cách sắp xếp mà **không phần tử nào ở vị trí cũ**" → derangement. Dãy `0, 1, 2, 9, 44, 265, ...`. Hãy nhớ bốn con đường: truy hồi hai số hạng, truy hồi một số hạng `D(n) = n D(n - 1) + (-1)^n`, bao hàm – loại trừ, và `round(n!/e)` (chỉ để kiểm tra nhanh `n` nhỏ).
- **Đếm thay vì liệt kê:** khi đề hỏi "bao nhiêu cách" và `n` lớn, đừng sinh các cấu hình; tìm **công thức truy hồi** bằng cách xét "phần tử đầu tiên đi đâu" rồi quy về bài nhỏ hơn.
- **Modulo:** lấy `% mod` sau **mỗi** phép cộng/nhân. Python cho kết quả không âm với `% mod`, nhưng ở C++/Java phải cộng thêm `mod` khi trừ.
- **Trường hợp cơ sở "rỗng":** `f(0) = 1` thường cần để truy hồi đúng (tích rỗng/cách duy nhất không làm gì).
- **Tối ưu bộ nhớ DP:** khi `f[i]` chỉ phụ thuộc vài giá trị gần nhất thì giữ vài biến thay cho cả bảng.
- **Gán đồng thời** `a, b = b, ...` tránh bug thứ tự.
- **Kiểm tra bằng dãy số đã biết:** dãy `1, 0, 1, 2, 9, 44, 265, 1854, ...` (A000166) là oracle tin cậy cho các `n` nhỏ.
- **Họ bài liên quan:** Fibonacci/Climbing Stairs (truy hồi hai số hạng), số Catalan, đếm hoán vị có ràng buộc vị trí.

## 15. Biến thể

1. **Số hoán vị có đúng `k` điểm cố định** (rencontres numbers): `R(n, k) = C(n, k) * D(n - k)` — chọn `k` điểm cố định, phần còn lại lệch hết.
2. **Xác suất một hoán vị ngẫu nhiên là derangement:** `D(n) / n! -> 1/e ≈ 0,3679` (bài toán "phát nhầm mũ" — hat-check).
3. **Derangement của mảng có phần tử lặp / có vị trí cấm tổng quát** (đa thức rook, ma trận cấm): dùng bao hàm – loại trừ tổng quát.
4. **Liệt kê** (không đếm) mọi derangement với `n` nhỏ.
5. **Derangement có thêm điều kiện** (không có hai phần tử kề nhau giữ nguyên thứ tự; ma trận hoán vị `0/1` có đường chéo 0...).
6. **Cùng họ truy hồi:** số cách ghép cặp, số Catalan, số Bell.

Demo biến thể 1 và 2 (đối chiếu với vét cạn):

In [35]:
def rencontres(n, k):
    """Số hoán vị của n phần tử có ĐÚNG k điểm cố định (số chính xác, n nhỏ)."""
    if k > n:
        return 0
    return comb(n, k) * exact_two[n - k]


def count_fixed_brute(n, k):
    return sum(1 for p in permutations(range(1, n + 1)) if sum(x == i + 1 for i, x in enumerate(p)) == k)


for n in range(1, 8):
    row = [rencontres(n, k) for k in range(n + 1)]
    assert sum(row) == factorial(n)                      # mọi hoán vị có k điểm cố định nào đó
    for k in range(n + 1):
        assert row[k] == count_fixed_brute(n, k), (n, k)
    print(f"n = {n}: số hoán vị có đúng k = 0..{n} điểm cố định: {row}")

print()
for n in (3, 5, 8, 10, 12, 15):
    ratio = Fraction(exact_two[n], factorial(n))
    print(f"n = {n:>2}: D(n)/n! = {float(ratio):.6f}")
print("1/e         =", f"{1 / EULER:.6f}")

n = 1: số hoán vị có đúng k = 0..1 điểm cố định: [0, 1]
n = 2: số hoán vị có đúng k = 0..2 điểm cố định: [1, 0, 1]
n = 3: số hoán vị có đúng k = 0..3 điểm cố định: [2, 3, 0, 1]
n = 4: số hoán vị có đúng k = 0..4 điểm cố định: [9, 8, 6, 0, 1]
n = 5: số hoán vị có đúng k = 0..5 điểm cố định: [44, 45, 20, 10, 0, 1]
n = 6: số hoán vị có đúng k = 0..6 điểm cố định: [265, 264, 135, 40, 15, 0, 1]


n = 7: số hoán vị có đúng k = 0..7 điểm cố định: [1854, 1855, 924, 315, 70, 21, 0, 1]

n =  3: D(n)/n! = 0.333333
n =  5: D(n)/n! = 0.366667
n =  8: D(n)/n! = 0.367882
n = 10: D(n)/n! = 0.367879
n = 12: D(n)/n! = 0.367879
n = 15: D(n)/n! = 0.367879
1/e         = 0.367879


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê hoán vị | Định nghĩa | `O(n * n!)` | `O(n)` | Chỉ để kiểm chứng `n <= 8` |
| 2. Quay lui cắt tỉa | Xây từng vị trí, bỏ nhánh sai | `O(n!)` | `O(n)` | Khi cần **liệt kê** hoặc `n <= 10` |
| 3. Bao hàm – loại trừ | `Σ (-1)^k n!/k!` | `O(n)` | `O(1)` | Khi nghĩ theo tổ hợp, mở rộng cho ràng buộc khác |
| 4. DP hai số hạng | `f[i] = (i-1)(f[i-1] + f[i-2])` | `O(n)` | `O(1)` (hai biến) | **Cách nộp**: ý nghĩa tổ hợp rõ |
| 5. Công thức một số hạng | `f[i] = i f[i-1] + (-1)^i` | `O(n)` | `O(1)` | Ngắn nhất; nhớ dễ nhưng ít trực giác |

Về ý tưởng: cách 1 và 2 **liệt kê**; cách 3 **đếm gián tiếp** bằng loại trừ; cách 4 và 5 **đếm bằng truy hồi** (cách 4 theo vị trí của số 1, cách 5 là dạng rút gọn đại số).

In [36]:
# so sánh thời gian: n = 8 (cả ba hướng), rồi n = 10^6 (chỉ các cách tuyến tính)
n = 8
for name, f in (("vét cạn", find_derangement_brute), ("cách 2", Solution2().findDerangement),
                ("cách 3", Solution3().findDerangement), ("cách 4", Solution4().findDerangement),
                ("cách 5", Solution5().findDerangement)):
    t0 = time.perf_counter()
    r = f(n)
    print(f"n = {n}: {name:8} -> {r} trong {(time.perf_counter() - t0) * 1000:8.3f} ms")

n = 10**6
ref = Solution().findDerangement(n)
for name, f in (("cách 3", Solution3().findDerangement), ("cách 4", Solution4().findDerangement),
                ("cách 5", Solution5().findDerangement)):
    t0 = time.perf_counter()
    r = f(n)
    assert r == ref
    print(f"n = 10^6: {name} -> {r} trong {(time.perf_counter() - t0) * 1000:8.1f} ms")

n = 8: vét cạn  -> 14833 trong   33.648 ms
n = 8: cách 2   -> 14833 trong   19.599 ms
n = 8: cách 3   -> 14833 trong    0.007 ms
n = 8: cách 4   -> 14833 trong    0.004 ms
n = 8: cách 5   -> 14833 trong    0.004 ms


n = 10^6: cách 3 -> 102701088 trong    137.4 ms
n = 10^6: cách 4 -> 102701088 trong    140.6 ms


n = 10^6: cách 5 -> 102701088 trong    108.7 ms


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [37]:
print([Solution().findDerangement(n) for n in range(1, 8)])
print(Solution().findDerangement(13))

[0, 1, 2, 9, 44, 265, 1854]
290792918


**Bài 2 (tính tay).** (a) Dùng công thức truy hồi `D(n) = (n - 1)(D(n - 1) + D(n - 2))` tính `D(5)` và `D(6)` từ `D(3) = 2`, `D(4) = 9`. Gán `ans2a = D(5)`, `ans2b = D(6)` (đang là `None`). (b) Có bao nhiêu hoán vị của `[1,2,3,4]` có **đúng một** điểm cố định? Gán `ans2c`.

In [38]:
ans2a = None   # TODO: D(5)
ans2b = None   # TODO: D(6)
ans2c = None   # TODO: số hoán vị của [1,2,3,4] có đúng 1 điểm cố định

**Bài 3 (điền TODO).** Hoàn thành `derange_one_term(n)` bằng công thức một số hạng `f(i) = i * f(i - 1) + (-1)^i` (số chính xác, **không modulo**, `n` nhỏ).

In [39]:
def derange_one_term(n):
    # TODO: f = 1 (ứng với D(0)); lặp i = 1..n
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_derangement(n)` (có modulo `10^9 + 7`) theo cách DP hai số hạng với hai biến.

In [40]:
def my_derangement(n):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `count_with_k_fixed(n, k)`: số hoán vị của `[1..n]` có **đúng `k`** điểm cố định, theo modulo `10^9 + 7`, cho `n` đến `2000` (dùng `D(n - k)` và tổ hợp `C(n, k)`; có thể dùng `math.comb` rồi `% MOD`). (b) Viết `prob_derangement(n)`: xác suất một hoán vị ngẫu nhiên của `n` phần tử là derangement, dưới dạng `Fraction` chính xác (`D(n) / n!`), cho `n` đến `12`; sau đó kiểm tra nó gần `1/e` đến sai số `10^-6` khi `n = 12`.

In [41]:
def count_with_k_fixed(n, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dãy `D(1..7)` là `0, 1, 2, 9, 44, 265, 1854`. Với `n = 13` hãy nhớ rằng đáp án bị rút gọn modulo.
- Bài 2: (a) áp dụng công thức truy hồi hai lần. (b) chọn điểm cố định, phần còn lại phải lệch hết.
- Bài 3: bắt đầu `f = 1`; dấu là `+1` nếu `i` chẵn, `-1` nếu `i` lẻ.
- Bài 4: `a, b = 1, 0`; lặp từ `i = 2`.
- Bài 5: (a) dùng `R(n, k) = C(n, k) * D(n - k)`; (b) `Fraction(D, factorial(n))`.

**Mức 2 (vừa):**
- Bài 2: `D(5) = 4 * (9 + 2) = 44`; `D(6) = 5 * (44 + 9) = 265`. (c) có `4` cách chọn điểm cố định, 3 phần tử còn lại lệch hết `D(3) = 2` cách.
- Bài 3: `f = i * f + (1 if i % 2 == 0 else -1)`.
- Bài 4: `a, b = b, (i - 1) * (a + b) % mod` và `return b if n >= 1 else a`.
- Bài 5(a): tính trước mảng `D[0..n]` theo modulo, rồi `comb(n, k) % MOD * D[n - k] % MOD`.

**Mức 3 (gần đáp án):**
- Bài 3: xem mục 7.4.
- Bài 4: xem mục 9, code nộp.
- Bài 5: (a) `D = [1, 0]`; `for i in range(2, n + 1): D.append((i - 1) * (D[-1] + D[-2]) % MOD)`; trả `comb(n, k) % MOD * D[n - k] % MOD` (nếu `k > n` trả 0). (b) `Fraction(D[n], factorial(n))` với `D` tính chính xác (không modulo).

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [42]:
# Bài 1
assert [Solution().findDerangement(n) for n in range(1, 8)] == [0, 1, 2, 9, 44, 265, 1854]
assert Solution().findDerangement(13) == 2290792932 % MOD == 290792918

# Bài 2
ans2a = 44
ans2b = 265
ans2c = 4 * 2
assert ans2a == 4 * (9 + 2) == KNOWN[5]
assert ans2b == 5 * (44 + 9) == KNOWN[6]
assert ans2c == count_fixed_brute(4, 1) == rencontres(4, 1)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [43]:
# Bài 3
def derange_one_term(n):
    f = 1
    for i in range(1, n + 1):
        f = i * f + (1 if i % 2 == 0 else -1)
    return f

assert [derange_one_term(n) for n in range(16)] == KNOWN
assert derange_one_term(100) == exact_two[100]
print("Bài 3 OK")

Bài 3 OK


In [44]:
# Bài 4
def my_derangement(n):
    mod = 10**9 + 7
    a, b = 1, 0
    for i in range(2, n + 1):
        a, b = b, (i - 1) * (a + b) % mod
    return b if n >= 1 else a

for n in range(1, 400):
    assert my_derangement(n) == exact_two[n] % MOD, n
rng = random.Random(4)
for _ in range(5):
    n = rng.randint(1, 10**6)
    assert my_derangement(n) == Solution().findDerangement(n)
print("Bài 4 OK")

Bài 4 OK


In [45]:
# Bài 5 (a)
def count_with_k_fixed(n, k):
    if k > n:
        return 0
    D = [1, 0]
    for i in range(2, n - k + 1):
        D.append((i - 1) * (D[-1] + D[-2]) % MOD)
    return comb(n, k) % MOD * D[n - k] % MOD


for n in range(0, 8):
    for k in range(0, n + 2):
        want = count_fixed_brute(n, k) if n >= 1 else (1 if k == 0 else 0)
        assert count_with_k_fixed(n, k) == want % MOD, (n, k)
assert count_with_k_fixed(5, 5) == 1            # chỉ hoán vị đơn vị
assert count_with_k_fixed(5, 4) == 0            # không thể có đúng n - 1 điểm cố định
assert sum(count_with_k_fixed(60, k) for k in range(61)) % MOD == factorial(60) % MOD

# Bài 5 (b)
def prob_derangement(n):
    d = [1, 0]
    for i in range(2, n + 1):
        d.append((i - 1) * (d[-1] + d[-2]))
    return Fraction(d[n], factorial(n))

assert prob_derangement(1) == 0
assert prob_derangement(2) == Fraction(1, 2)
assert prob_derangement(3) == Fraction(1, 3)
assert prob_derangement(4) == Fraction(3, 8)
assert abs(float(prob_derangement(12)) - 1 / EULER) < 1e-6
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP hai số hạng (nộp):**

```python
class Solution:
    def findDerangement(self, n: int) -> int:
        mod = 10**9 + 7
        a, b = 1, 0                    # f[0], f[1]
        for i in range(2, n + 1):
            a, b = b, (i - 1) * (a + b) % mod
        return b if n >= 1 else a
```

**Template một số hạng:**

```python
f = 1
for i in range(1, n + 1):
    f = (i * f + (1 if i % 2 == 0 else -1)) % mod
return f
```

**Công thức cần nhớ:**

- `D(0) = 1`, `D(1) = 0`, `D(n) = (n - 1)(D(n - 1) + D(n - 2))`
- `D(n) = n * D(n - 1) + (-1)^n`
- `D(n) = Σ_{k=0}^{n} (-1)^k n!/k!`
- Dãy: `1, 0, 1, 2, 9, 44, 265, 1854, 14833, ...`

**Checklist trước khi submit:**

- [ ] `f[0] = 1`, `f[1] = 0` đúng; `n = 1` trả `0`?
- [ ] Hệ số là `(i - 1)`, không phải `i`?
- [ ] Lấy `% mod` sau **mỗi** phép nhân/cộng (và mod = `10^9 + 7`)?
- [ ] Cập nhật hai biến bằng gán đồng thời `a, b = b, ...`?
- [ ] Dùng vòng `for` (không đệ quy sâu `10^6`); `O(n)` thời gian là đủ?
- [ ] Đã so với dãy đã biết `0, 1, 2, 9, 44, 265, ...` và với vét cạn khi `n <= 8`?